# 01 - Data Acquisition

This is the original HIV-ESM-2 data acquisition notebook. It remains the first step in the single consolidated workflow and prepares raw Stanford HIVDB files for downstream preprocessing.

CPU notebook: GPU is not required.


In [1]:
# Colab/local project setup — see COLAB_SETUP.md
import sys
from pathlib import Path

_nb = Path.cwd().resolve()
_notebooks = _nb if (_nb / 'notebook_setup.py').exists() else _nb / 'notebooks'
if str(_notebooks) not in sys.path:
    sys.path.insert(0, str(_notebooks))

from notebook_setup import bootstrap_notebook_environment, export_notebook_globals

paths, pipeline_config = bootstrap_notebook_environment(
    install_deps=True,
    mount_drive=True,
)
globals().update(export_notebook_globals(paths, pipeline_config))

print('Project root:', PROJECT_ROOT)
print('Running in Colab:', IN_COLAB)
print('ENABLE_IMPROVED_PIPELINE:', ENABLE_IMPROVED_PIPELINE)


Project root: /home/niranjanr/HIV-ESM2-Enhanced
Running in Colab: False
ENABLE_IMPROVED_PIPELINE: True



---

## Original HIV-ESM-2 Acquisition Workflow

The cells below are preserved from the original acquisition notebook.


In [2]:
# Notebook 01: Data Acquisition
#
# HIV Drug Resistance Prediction with ESM-2
#
# ---
#
# Objective: Download and parse Stanford HIVDB sequence and phenotype data.
#
# Data Source: Stanford HIV Drug Resistance Database (HIVDB)
# - https://hivdb.stanford.edu/
#
# Expected Output:
# - 6,308 sequences (2,171 PI, 1,867 NRTI, 2,270 NNRTI)
# - Resistance annotations for 18 drugs
#
# ---

In [3]:
# Install dependencies
# !pip install -q pandas numpy biopython tqdm

In [4]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_processing import (
    get_drug_list, load_fasta, save_fasta,
    parse_hivdb_sequences, get_dataset_statistics,
    reconstruct_sequences_from_positions,
    HIV_PROTEASE_REFERENCE,
    HIV_RT_REFERENCE
)

print("Imports complete")



Imports complete


/home/niranjanr/miniconda3/envs/tfgpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
DATA_RAW = DATA_DIR / 'raw'
DATA_PROCESSED = DATA_DIR / 'processed'

# Create directories
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: /home/niranjanr/HIV-ESM2-Enhanced
Data directory: /home/niranjanr/HIV-ESM2-Enhanced/data


## 1. Download HIVDB Data

Stanford HIVDB provides genotype-phenotype datasets for download.

**Manual download instructions:**
1. Visit https://hivdb.stanford.edu/pages/genopheno.dataset.html
2. Download the genotype-phenotype datasets for:
   - Protease Inhibitors (PI)
   - NRTIs
   - NNRTIs
3. Place files in `data/raw/`

In [6]:
# Check for downloaded data
expected_files = [
    'PI_DataSet.txt',
    'NRTI_DataSet.txt', 
    'NNRTI_DataSet.txt'
]

print("Checking for HIVDB data files...")
for filename in expected_files:
    filepath = DATA_RAW / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / 1e6
        print(f"  Found: {filename} ({size_mb:.2f} MB)")
    else:
        print(f"  Missing: {filename}")
        print(f"    Download from https://hivdb.stanford.edu/pages/genopheno.dataset.html")

Checking for HIVDB data files...
  Found: PI_DataSet.txt (0.65 MB)
  Found: NRTI_DataSet.txt (1.16 MB)
  Found: NNRTI_DataSet.txt (1.73 MB)


## 2. Parse HIVDB Data

Extract sequences and phenotype labels from HIVDB format.

In [7]:
def parse_hivdb_dataset(filepath):
    """
    Parse Stanford HIVDB genotype-phenotype dataset.
    
    The format is a tab-separated file with:
    - Sequence information
    - Drug resistance fold-change values
    - Resistance classifications
    """
    df = pd.read_csv(filepath, sep='\t', low_memory=False)
    print(f"  Loaded {len(df)} records")
    print(f"  Columns: {df.columns.tolist()[:10]}...")
    return df

In [8]:
# Parse each drug class
datasets = {}

for drug_class in ['PI', 'NRTI', 'NNRTI']:
    filepath = DATA_RAW / f'{drug_class}_DataSet.txt'
    
    if filepath.exists():
        print(f"\nParsing {drug_class}...")
        datasets[drug_class] = parse_hivdb_dataset(filepath)
    else:
        print(f"\nSkipping {drug_class} (file not found)")



Parsing PI...
  Loaded 2171 records
  Columns: ['SeqID', 'FPV', 'ATV', 'IDV', 'LPV', 'NFV', 'SQV', 'TPV', 'DRV', 'P1']...

Parsing NRTI...
  Loaded 1867 records
  Columns: ['SeqID', '3TC', 'ABC', 'AZT', 'D4T', 'DDI', 'TDF', 'P1', 'P2', 'P3']...

Parsing NNRTI...


  Loaded 2272 records
  Columns: ['SeqID', 'EFV', 'NVP', 'ETR', 'RPV', 'DOR', 'P1', 'P2', 'P3', 'P4']...


## 3. Extract Sequences and Phenotypes

Separate sequence data from resistance phenotypes.

In [9]:
def extract_sequences_and_phenotypes(df, drug_class):
    """
    Extract sequences and phenotype labels from HIVDB dataset.
    """
    # Common sequence column names in HIVDB
    seq_cols = ['SeqID', 'IsolateID', 'Subtype', 'AASequence', 'NASequence']

    # Identify sequence column
    aa_col = None
    for col in df.columns:
        if 'AA' in col.upper() and 'SEQ' in col.upper():
            aa_col = col
            break
        elif 'SEQUENCE' in col.upper():
            aa_col = col

    if aa_col is None:
        # Try reconstructing sequences from position columns (P1, P2, ...)
        pos_cols = [c for c in df.columns if c.startswith('P') and c[1:].isdigit()]
        if pos_cols:
            reference = HIV_PROTEASE_REFERENCE if drug_class == 'PI' else HIV_RT_REFERENCE
            sequences = reconstruct_sequences_from_positions(df, reference)
        else:
            print(f"  Warning: Could not find sequence column or position columns")
            return None, None, None

    # Get drug-specific columns (fold-change and classification)
    drugs = get_drug_list(drug_class)

    phenotype_cols = []
    for drug in drugs:
        if drug in df.columns:
            phenotype_cols.append(drug)
        for suffix in ['_FC', '_class3', '_class2']:
            col = f"{drug}{suffix}"
            if col in df.columns:
                phenotype_cols.append(col)

    sequences = df[aa_col].tolist() if aa_col is not None else sequences
    seq_ids = df['SeqID'].tolist() if 'SeqID' in df.columns else [f"seq_{i}" for i in range(len(df))]
    phenotypes = df[phenotype_cols].copy() if phenotype_cols else __import__('pandas').DataFrame()
    phenotypes['seq_id'] = seq_ids

    print(f"  Extracted {len(sequences)} sequences")
    print(f"  Phenotype columns: {len(phenotype_cols)}")

    return sequences, seq_ids, phenotypes



In [10]:
# Extract sequences and phenotypes for each drug class
processed_data = {}

for drug_class, df in datasets.items():
    print(f"\nProcessing {drug_class}...")
    sequences, seq_ids, phenotypes = extract_sequences_and_phenotypes(df, drug_class)
    
    if sequences:
        # Filter sequences (remove those with gaps or unknown amino acids)
        valid_mask = [all(c.isalpha() and c != 'X' for c in seq.replace('-', '')) 
                     for seq in sequences]
        
        valid_sequences = [s for s, v in zip(sequences, valid_mask) if v]
        valid_seq_ids = [s for s, v in zip(seq_ids, valid_mask) if v]
        valid_phenotypes = phenotypes[valid_mask].reset_index(drop=True)
        
        processed_data[drug_class] = {
            'sequences': valid_sequences,
            'seq_ids': valid_seq_ids,
            'phenotypes': valid_phenotypes
        }
        
        print(f"  After filtering: {len(valid_sequences)} sequences")


Processing PI...


  Extracted 2171 sequences
  Phenotype columns: 8
  After filtering: 2086 sequences

Processing NRTI...


  Extracted 1867 sequences
  Phenotype columns: 6
  After filtering: 1714 sequences

Processing NNRTI...


  Extracted 2272 sequences
  Phenotype columns: 4
  After filtering: 2066 sequences


## 4. Dataset Statistics

Compute summary statistics for the processed data.

In [11]:
print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

total_seqs = 0

for drug_class, data in processed_data.items():
    n_seqs = len(data['sequences'])
    total_seqs += n_seqs
    
    seq_lengths = [len(s) for s in data['sequences']]
    
    print(f"\n{drug_class}:")
    print(f"  Sequences: {n_seqs:,}")
    print(f"  Sequence length: {np.mean(seq_lengths):.0f} +/- {np.std(seq_lengths):.0f} aa")
    print(f"  Range: {np.min(seq_lengths)}-{np.max(seq_lengths)} aa")
    
    # Drug-specific statistics
    drugs = get_drug_list(drug_class)
    print(f"  Drugs ({len(drugs)}): {drugs}")

print(f"\nTotal sequences: {total_seqs:,}")



DATASET STATISTICS

PI:
  Sequences: 2,086
  Sequence length: 99 +/- 0 aa
  Range: 99-99 aa
  Drugs (8): ['ATV', 'DRV', 'FPV', 'IDV', 'LPV', 'NFV', 'SQV', 'TPV']

NRTI:
  Sequences: 1,714
  Sequence length: 240 +/- 0 aa
  Range: 240-240 aa
  Drugs (6): ['ABC', 'AZT', 'D4T', 'DDI', '3TC', 'TDF']

NNRTI:
  Sequences: 2,066
  Sequence length: 240 +/- 0 aa
  Range: 240-240 aa
  Drugs (4): ['EFV', 'ETR', 'NVP', 'RPV']

Total sequences: 5,866


In [12]:
print("\n" + "="*60)
print("RESISTANCE PREVALENCE BY DRUG")
print("="*60)

for drug_class, data in processed_data.items():
    phenotypes = data["phenotypes"]
    drugs = get_drug_list(drug_class)
    print(f"\n{drug_class}:")
    
    for drug in drugs:
        class2_col = f"{drug}_class2"
        if class2_col in phenotypes.columns:
            resistant = (phenotypes[class2_col] >= 2).sum()
        else:
            if drug in phenotypes.columns:
                drug_values = pd.to_numeric(phenotypes[drug], errors="coerce")
                resistant = (drug_values >= 2.5).sum()
            else:
                resistant = 0
        
        total = len(phenotypes)
        pct = 100.0 * resistant / total if total > 0 else 0
        print(f"  {drug}: {resistant}/{total} ({pct:.1f}%)")



RESISTANCE PREVALENCE BY DRUG

PI:
  ATV: 791/2086 (37.9%)
  DRV: 311/2086 (14.9%)
  FPV: 902/2086 (43.2%)
  IDV: 1045/2086 (50.1%)
  LPV: 934/2086 (44.8%)
  NFV: 1231/2086 (59.0%)
  SQV: 911/2086 (43.7%)
  TPV: 338/2086 (16.2%)

NRTI:
  ABC: 944/1714 (55.1%)
  AZT: 793/1714 (46.3%)
  D4T: 397/1714 (23.2%)
  DDI: 250/1714 (14.6%)
  3TC: 1096/1714 (63.9%)
  TDF: 217/1714 (12.7%)

NNRTI:
  EFV: 927/2066 (44.9%)
  ETR: 293/2066 (14.2%)
  NVP: 965/2066 (46.7%)
  RPV: 133/2066 (6.4%)


## 5. Save Processed Data

Save sequences and phenotypes for downstream analysis.

In [13]:
print("\nSaving processed data (preserving existing non-empty files)...")
for drug_class, data in processed_data.items():
    fasta_path = DATA_PROCESSED / f'{drug_class}_sequences.fasta'
    pheno_path = DATA_PROCESSED / f'{drug_class}_phenotypes.csv'
    if not fasta_path.exists() or fasta_path.stat().st_size == 0:
        save_fasta(data['sequences'], data['seq_ids'], fasta_path)
        print(f'Wrote {fasta_path.name}')
    else:
        print(f'Reusing {fasta_path.name}')
    if not pheno_path.exists() or pheno_path.stat().st_size == 0:
        data['phenotypes'].to_csv(pheno_path, index=False)
        print(f'Wrote {pheno_path.name}')
    else:
        print(f'Reusing {pheno_path.name}')
import json
metadata_path = DATA_PROCESSED / 'metadata.json'
if not metadata_path.exists() or metadata_path.stat().st_size == 0:
    metadata = {'date_processed': pd.Timestamp.now().isoformat(),
                'drug_classes': list(processed_data.keys()),
                'counts': {dc: len(d['sequences']) for dc, d in processed_data.items()},
                'total_sequences': total_seqs}
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
else:
    print('Reusing metadata.json')
print('Done!')



Saving processed data (preserving existing non-empty files)...
Reusing PI_sequences.fasta
Reusing PI_phenotypes.csv
Reusing NRTI_sequences.fasta
Reusing NRTI_phenotypes.csv
Reusing NNRTI_sequences.fasta
Reusing NNRTI_phenotypes.csv
Reusing metadata.json
Done!


## Summary

**Expected results:**
- PI: 2,171 sequences, 8 drugs
- NRTI: 1,867 sequences, 6 drugs
- NNRTI: 2,270 sequences, 4 drugs
- Total: 6,308 sequences, 18 drugs

**Next steps:**
- Notebook 02: Develop baseline model using binary mutation encoding
- Notebook 03: Extract ESM-2 embeddings

## Workflow Handoff

Next notebook: `02_data_preprocessing.ipynb`.
